# Incremental Data Loading using AutoLoader

In [0]:
%sql
CREATE SCHEMA netflix_catalog.net_schema;

In [0]:
# Create the Checkpoint location and schemaLocation
checkpoint_location = "abfss://bronze@netflixprojectnafis.dfs.core.windows.net/checkpoint_netflix_titles"

schema_location = "abfss://bronze@netflixprojectnafis.dfs.core.windows.net/schema_netflix_titles"

In [0]:
df = spark.readStream\
  .format("cloudFiles")\
  .option("cloudFiles.format", "csv")\
  .option("cloudFiles.schemaLocation", schema_location)\
  .load("abfss://raw@netflixprojectnafis.dfs.core.windows.net/")

In [0]:
# display(df)
# Never display before write stream as it will cause a mini stream and later the write stream will consider it as already read via the checkpoints and it wil be skipped

In [0]:
# Writing the data to bronze container

query = df.writeStream\
    .format("delta")\
    .option("checkpointLocation", checkpoint_location)\
    .option("mergeSchema", "true")\
    .trigger(processingTime="10 seconds")\
    .start("abfss://bronze@netflixprojectnafis.dfs.core.windows.net/netflix_titles")